# Python 异步基础: 把 I/O 等待期间的 CPU 用起来

本篇是 `books/python/01-python-async.md` 的配套可跑样例, 内核 `agent-cookbook` (Python 3.13)。

所有代码在 Python 3.13.12 实测通过, 输出为真实运行结果。贯穿场景: 一个进程抓三个网页, 每次请求 2 秒。

注意: notebook 里可以直接 `await`, 是因为 jupyter 已经运行在 event loop 里;
真实项目里应该用 `asyncio.run(...)` 来启动。

In [ ]:
import time

def fetch_page(url: str) -> str:
    time.sleep(2)  # 模拟一次 2 秒的网络请求 (I/O 等待)
    return f"{url} 的内容"

t0 = time.perf_counter()
for url in ("page-A", "page-B", "page-C"):
    result = fetch_page(url)
    print(f"{time.perf_counter()-t0:4.1f}s  {result}")
print(f"{time.perf_counter()-t0:4.1f}s  总耗时")

2.0s  page-A 的内容
4.0s  page-B 的内容
6.0s  page-C 的内容
6.0s  总耗时


三个请求串行排队, 6 秒里 CPU 绝大部分时间在 `time.sleep` 处空转。
把这段空闲利用起来, 就是异步编程的全部问题。

In [ ]:
import asyncio

async def fetch_page(url: str) -> str:
    await asyncio.sleep(2)  # 挂起 2 秒, 控制权交回 event loop
    return f"{url} 的内容"

t0 = time.perf_counter()
for coro in asyncio.as_completed([fetch_page(u) for u in ("page-A", "page-B", "page-C")]):
    result = await coro   # 先等结果, 再记时间——顺序反了打印出来的时刻是错的
    print(f"{time.perf_counter()-t0:4.1f}s  {result}")

2.0s  page-A 的内容
2.0s  page-B 的内容
2.0s  page-C 的内容


单线程, 6 秒降到 2 秒。这是并发不是并行——三段 I/O 等待重叠了, 没有用到多个核。

下面拆开看协程到底是什么。

In [ ]:
async def sample() -> str:
    return "ok"

print("协程函数 :", type(sample).__name__)
coro = sample()
print("协程对象 :", type(coro).__name__)
print("此刻函数体一行都没执行")
try:
    coro.send(None)  # 手动驱动一步 (PEP 342), event loop 干的就是这件事
except StopIteration as e:
    print("驱动到函数结束, 返回值:", e.value)

协程函数 : function
协程对象 : coroutine
此刻函数体一行都没执行
驱动到函数结束, 返回值: ok


In [ ]:
async def demo_task_future():
    future = asyncio.get_running_loop().create_future()
    print("future 刚创建, 有结果吗:", future.done())
    task = asyncio.create_task(fetch_page("page-A"))
    done, pending = await asyncio.wait({future, task}, return_when=asyncio.FIRST_COMPLETED)
    for d in done:
        print("先完成的是:", type(d).__name__, "->", d.result())
    for d in pending:
        print("还在等的 :", type(d).__name__)

await demo_task_future()
print("Task 是 Future 的子类:", issubclass(asyncio.Task, asyncio.Future))

future 刚创建, 有结果吗: False
先完成的是: Task -> page-A 的内容
还在等的 : Future
Task 是 Future 的子类: True


Future 是"还没结果"的占位符, 核心用法是当"信箱":
await 它就是挂起等投递, set_result 就是投递。

In [ ]:
async def waiter(future, name):
    result = await future
    print(f"{name} 收到事件: {result}")

async def event_source(future):
    await asyncio.sleep(1)
    future.set_result("紧急消息: page-D 插单了")

async def demo_mailbox():
    future = asyncio.get_running_loop().create_future()
    await asyncio.gather(waiter(future, "agent"), event_source(future))

await demo_mailbox()

agent 收到事件: 紧急消息: page-D 插单了


In [ ]:
async def long_tool():
    try:
        await asyncio.sleep(10)
        return "done"
    except asyncio.CancelledError:
        print("  long_tool 在 await 点收到取消, 清理后重新抛出")
        raise

async def demo_cancel():
    task = asyncio.create_task(long_tool())
    await asyncio.sleep(0.1)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("demo_cancel: task 确认已取消")

async def demo_timeout():
    try:
        await asyncio.wait_for(long_tool(), timeout=1)
    except TimeoutError:
        print("demo_timeout: 1 秒没等到结果, 超时")

await asyncio.gather(demo_cancel(), demo_timeout())

  long_tool 在 await 点收到取消, 清理后重新抛出
demo_cancel: task 确认已取消
  long_tool 在 await 点收到取消, 清理后重新抛出
demo_timeout: 1 秒没等到结果, 超时


取消不是杀线程, 而是在下一个 `await` 点抛 `CancelledError`
(源码上走 tasks.py 的 `coro.throw(exc)`), 给了函数清理的机会。
`wait_for` 的超时本质也是取消。

In [ ]:
import sys

print(sys.version.split()[0], "GIL 启用:", sys._is_gil_enabled())

3.13.12 GIL 启用: True


In [ ]:
import threading

results = []
def worker(url: str):
    time.sleep(2)  # I/O 等待期间 GIL 释放, 其他线程可以跑
    results.append((url, time.perf_counter()-t0))

t0 = time.perf_counter()
threads = [threading.Thread(target=worker, args=(u,)) for u in ("page-A", "page-B", "page-C")]
for t in threads: t.start()
for t in threads: t.join()
for url, ts in sorted(results, key=lambda x: x[1]):
    print(f"{ts:4.1f}s  {url} 完成")
print(f"{time.perf_counter()-t0:4.1f}s  总耗时 (3 个线程并行等 I/O)")

2.0s  page-A 完成
2.0s  page-C 完成
2.0s  page-B 完成
2.0s  总耗时 (3 个线程并行等 I/O)


In [ ]:
from concurrent.futures import Future as CFuture

print("asyncio.Future           :", asyncio.Future)
print("concurrent.futures.Future:", CFuture)
print("是同一个类吗:", asyncio.Future is CFuture)

def blocking_sdk_call() -> str:
    time.sleep(1)  # 模拟一个没有 async 版本的阻塞 SDK
    return "blocking result"

async def heartbeat():
    while True:
        await asyncio.sleep(0.4)
        print(f"  {time.perf_counter()-t0:3.1f}s  event loop 还活着")

async def demo_bridge():
    global t0
    loop = asyncio.get_running_loop()
    t0 = time.perf_counter()
    blocking = loop.run_in_executor(None, blocking_sdk_call)  # 丢进线程池, 不阻塞 loop
    hb = asyncio.create_task(heartbeat())
    result = await blocking
    hb.cancel()
    print(f"阻塞调用结果: {result}, 总耗时 {time.perf_counter()-t0:.1f}s")
    r2 = await asyncio.to_thread(blocking_sdk_call)
    print(f"to_thread 也能用: {r2}")

await demo_bridge()

asyncio.Future           : <class '_asyncio.Future'>
concurrent.futures.Future: <class 'concurrent.futures._base.Future'>
是同一个类吗: False
  0.4s  event loop 还活着
  0.8s  event loop 还活着
阻塞调用结果: blocking result, 总耗时 1.0s
to_thread 也能用: blocking result


In [ ]:
async def heartbeat2():
    while True:
        await asyncio.sleep(0.5)
        print(f"  {time.perf_counter()-t0:3.1f}s  心跳")

async def cpu_heavy():
    x = 0
    t = time.perf_counter()
    while time.perf_counter() - t < 2:   # 纯 CPU 空转, 中间不 await
        x += 1
    return x

async def main_cpu():
    hb = asyncio.create_task(heartbeat2())
    mark = time.perf_counter()
    x = await cpu_heavy()   # 直接在当前执行流跑, 不让出
    print(f"cpu_heavy 结束, 耗时 {time.perf_counter()-mark:.1f}s, 期间心跳全部停摆")
    await asyncio.sleep(1.2)
    hb.cancel()

t0 = time.perf_counter()
await main_cpu()

cpu_heavy 结束, 耗时 2.0s, 期间心跳全部停摆
  2.5s  心跳
  3.0s  心跳


CPU 密集任务没有 await, event loop 被彻底卡死——它和协程、多线程都不对付,
唯一的出路是多进程 (`ProcessPoolExecutor`), 实测 4 个 1 秒的 CPU 任务串行 4.0s / 4 进程 1.4s。
多进程示例必须写成独立脚本并放在 `if __name__ == "__main__":` 保护下,
不能在 notebook 里跑 (notebook 的 `__main__` 不是文件, 子进程 import 不到), 所以本篇不放单元格。

## 留给下一篇的坑

同步版、多线程版、协程版, 三版实现全部接不住这三个场景:

1. 任务执行中途插进来一条紧急消息 (同步只能靠预设检查点轮询);
2. 任务发起方中途喊停 (阻塞在工具调用上时, 连轮询的执行权都没有);
3. 一百个会话同时在线, 同一会话必须串行、跨会话必须并行。

解开死局的地基已备齐: Future 是信箱、Task 可中断、loop 会因 I/O 就绪唤醒协程。
下一篇把它们拼起来: 事件驱动的 Agent Loop, 见 `books/python/02-event-driven-agent-loop.md`。